In [7]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision.transforms import AutoAugment, AutoAugmentPolicy 

PRECOMPUTED_DIR = '/kaggle/working/precomputed_rois' # The location to save processed images

# --- 1. CONFIGURAZIONE ---
def find_paths():
    base_train = '/kaggle/input/dataset'
    # Cerca la cartella corretta per il test (che ha lo spazio nel nome)
    test_root = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.endswith("test-data/test_data"):
            test_root = root
            break
            
    if test_root is None:
        raise ValueError("Impossibile trovare la cartella test_data/images")

    return {
        'train_img': os.path.join(base_train, 'images'),
        'train_mask': os.path.join(base_train, 'masks'),
        'labels': os.path.join(base_train, 'train_labels.csv'),
        'test_img': test_root,
        'test_mask': test_root.replace('images', 'masks')
    }

PATHS = find_paths()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Iperparametri Ottimizzati
IMG_SIZE = 256
BATCH_SIZE = 32
N_FOLDS = 5
EPOCHS = 25
LR = 1e-4 
MIXUP_ALPHA = 0.4  # Intensità del MixUp

# --- 2. DATASET & PREPROCESSING ---
class ReinhardNormalizer:
    """Normalizzazione Colore per standardizzare i vetrini"""
    def __init__(self):
        self.target_mean = np.array([148.60, 169.30, 105.47])
        self.target_std = np.array([41.56, 11.23, 14.39])

    def __call__(self, img_rgb):
        try:
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype("float32")
            img_mean = np.mean(img_lab, axis=(0,1))
            img_std = np.std(img_lab, axis=(0,1)) + 1e-5
            img_lab = (img_lab - img_mean) / img_std
            img_lab = img_lab * self.target_std + self.target_mean
            img_lab = np.clip(img_lab, 0, 255).astype("uint8")
            return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
        except:
            return img_rgb

reinhard = ReinhardNormalizer()

def get_bbox_from_mask(mask):
    if np.sum(mask) == 0: return None
    rows, cols = np.where(mask > 0)
    return np.min(rows), np.max(rows), np.min(cols), np.max(cols)

# class TissueDataset(Dataset):
#     def __init__(self, dataframe, img_dir, mask_dir, transform=None, is_test=False):
#         self.df = dataframe
#         self.img_dir = img_dir
#         self.mask_dir = mask_dir
#         self.transform = transform
#         self.is_test = is_test

#     def __len__(self): return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         img_name = row['sample_index'] if not self.is_test else row['filename']
#         mask_name = img_name.replace("img", "mask")
        
#         img_path = os.path.join(self.img_dir, img_name)
#         mask_path = os.path.join(self.mask_dir, mask_name)
        
#         try:
#             image = cv2.imread(img_path)
#             image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#             mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
#         except:
#             image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
#             mask = None

#         # ROI Extraction
#         if mask is not None:
#             bbox = get_bbox_from_mask(mask)
#             if bbox:
#                 y_min, y_max, x_min, x_max = bbox
#                 image = image[y_min:y_max+1, x_min:x_max+1]
        
#         # Color Norm
#         image = reinhard(image)
#         image = Image.fromarray(image)
        
#         if self.transform: image = self.transform(image)
        
#         if self.is_test: return image, img_name
#         else: return image, torch.tensor(row['label_encoded'], dtype=torch.long)

# --- 2. DATASET & PREPROCESSING (Modified TissueDataset) ---
# ... (ReinhardNormalizer and get_bbox_from_mask remain the same) ...

class TissueDataset(Dataset):
    def __init__(self, dataframe, precomputed_dir, transform=None, is_test=False):
        self.df = dataframe
        # This directory now contains the final, preprocessed images
        self.precomputed_dir = precomputed_dir 
        self.transform = transform
        self.is_test = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['sample_index'] if not self.is_test else row['filename']
        
        # --- FAST READ FROM PRECOMPUTED FILE ---
        img_path = os.path.join(self.precomputed_dir, img_name)
        
        try:
            # Open the pre-processed image (now already cropped, resized, and normalized)
            image = Image.open(img_path).convert("RGB")
        except:
            # Fallback for failed reads (should be rare if precomputation was successful)
            # Returns a tensor of zeros to prevent training crash
            image = torch.zeros(3, IMG_SIZE, IMG_SIZE)
            if self.is_test: return image, img_name
            else: return image, torch.tensor(row['label_encoded'], dtype=torch.long)
            

        # Apply only the PyTorch/AutoAugment transforms (fast and GPU-friendly)
        if self.transform: image = self.transform(image)
        
        if self.is_test: return image, img_name
        else: return image, torch.tensor(row['label_encoded'], dtype=torch.long)

# --- 3. TRANSFORMS & MIXUP ---
def get_transforms():
    train_trans = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), 
        
        # *** AutoAugment for stronger generalization ***
        AutoAugment(policy=AutoAugmentPolicy.IMAGENET),
        
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_trans = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return train_trans, val_trans

def mixup_data(x, y, alpha=1.0):
    '''Returns mixed inputs, pairs of targets, and lambda'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# --- 4. MODELLO: EfficientNet-B0 (Pretrained) ---
def get_model(num_classes):
    # Usiamo B0 perché con 581 immagini, B3 o superiori overfittano troppo velocemente
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    
    # Custom Head
    in_features = model.classifier[1].in_features
    
    # EfficientNet already has a Dropout layer (0.2) before this.
    model.classifier = nn.Linear(in_features, num_classes) # num_classes should be 1
    
    return model.to(DEVICE)

# --- 4.5 PRECOMPUTATION FUNCTION ---

def preprocess_and_save_images(df, paths, output_dir, img_size):
    """
    Applies ROI extraction and Reinhard normalization once, saving the result 
    to a new output_dir so the DataLoader only handles fast reads and PyTorch transforms.
    """
    print("Starting precomputation and saving of processed ROIs...")
    
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize the Reinhard Normalizer
    reinhard = ReinhardNormalizer()
    
    # We will track which images were successfully processed
    processed_indices = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing Images"):
        img_name = row['sample_index']
        mask_name = img_name.replace("img", "mask")
        
        img_path = os.path.join(paths['train_img'], img_name)
        mask_path = os.path.join(paths['train_mask'], mask_name)
        output_path = os.path.join(output_dir, img_name)
        
        try:
            # 1. Read and Convert
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            # 2. ROI Extraction (using mask)
            if mask is not None:
                bbox = get_bbox_from_mask(mask)
                if bbox:
                    y_min, y_max, x_min, x_max = bbox
                    image = image[y_min:y_max+1, x_min:x_max+1]
            
            # 3. Color Normalization
            image = reinhard(image)
            
            # 4. Resize and Save (as PNG for better quality than JPEG)
            # Use PIL for robust and standard resizing for saving
            image_pil = Image.fromarray(image)
            # Resize the cropped ROI to the target size before saving
            image_resized = image_pil.resize((img_size, img_size), Image.Resampling.LANCZOS)
            
            image_resized.save(output_path, "PNG")
            processed_indices.append(idx)
            
        except Exception as e:
            # Handle cases where image/mask read fails
            print(f"Skipping image {img_name} due to error: {e}")
            continue

    # Return the filtered dataframe containing only successfully processed images
    return df.iloc[processed_indices].reset_index(drop=True)


def setup_data_and_preprocess():
    """
    Reads labels, performs binary grouping, runs the heavy CPU precomputation
    (ROI extraction and Reinhard normalization), and returns the filtered DataFrame.
    """
    print("--- 🧠 Data Setup & Preprocessing Started ---")
    
    # 1. Load Data and Filter by Existence
    df = pd.read_csv(PATHS['labels'])
    valid_mask = df['sample_index'].apply(lambda x: os.path.exists(os.path.join(PATHS['train_img'], x)))
    df = df[valid_mask].reset_index(drop=True)

    # 2. Binary Classification: NEW LABELING LOGIC
    GROUP_0 = ['Luminal A', 'Luminal B']
    GROUP_1 = ['HER2(+)', 'Triple negative']
    
    def get_binary_label(original_label):
        if original_label in GROUP_0:
            return 0
        elif original_label in GROUP_1:
            return 1
        else:
            return -1
            
    df['label_binary'] = df['label'].apply(get_binary_label)
    df['label_encoded'] = df['label_binary']
    
    # 3. Precompute ROIs, Resize, and Save (The CPU-heavy step)
    # This uses the preprocess_and_save_images function (defined in the previous answer)
    df_final = preprocess_and_save_images(df, PATHS, PRECOMPUTED_DIR, IMG_SIZE)
    
    print(f"--- ✅ Precomputation Complete. Final images available: {len(df_final)} ---")
    return df_final
    
# --- 5. TRAINING LOOP ---
def train_fold(fold, train_idx, val_idx, df, le, precomputed_dir):
    print(f"\n=== FOLD {fold+1}/{N_FOLDS} ===")
    
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    
    # Class Balancing
    class_counts = train_df['label_encoded'].value_counts().sort_index().values
    class_weights = 1. / class_counts
    sample_weights = [class_weights[l] for l in train_df['label_encoded']]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
    
    train_loader = DataLoader(TissueDataset(train_df, precomputed_dir, get_transforms()[0]), 
                              batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_loader = DataLoader(TissueDataset(val_df, precomputed_dir, get_transforms()[1]), 
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    model = get_model(num_classes = 1)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    history = {'t_loss': [], 'v_loss': [], 't_f1': [], 'v_f1': []}
    best_f1 = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        t_loss = 0
        t_preds, t_targets = [], []
        
        for imgs, lbls in tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            
            # --- MIXUP BLOCK ---
            lbls_float = lbls.float().unsqueeze(1)
            imgs, targets_a, targets_b, lam = mixup_data(imgs, lbls_float, MIXUP_ALPHA)
            out = model(imgs)
            loss = mixup_criterion(criterion, out, targets_a, targets_b, lam)
            
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
            
            # Per le metriche usiamo l'argmax approssimato (non perfetto col mixup ma indicativo)
            t_preds.extend((torch.sigmoid(out) >= 0.5).long().squeeze().cpu().numpy())
            t_targets.extend(targets_a.round().long().squeeze().cpu().numpy())

        scheduler.step()
        
        # Validation (No Mixup)
        model.eval()
        v_loss = 0
        v_preds, v_targets = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                lbls_float = lbls.float().unsqueeze(1)
                
                out = model(imgs)
                v_loss += criterion(out, lbls_float).item()
                v_preds.extend((torch.sigmoid(out) >= 0.5).long().squeeze().cpu().numpy())
                v_targets.extend(lbls.cpu().numpy())
        
        t_loss /= len(train_loader)
        v_loss /= len(val_loader)
        t_f1 = f1_score(t_targets, t_preds, average='macro') 
        v_f1 = f1_score(v_targets, v_preds, average='macro')
        
        history['t_loss'].append(t_loss); history['v_loss'].append(v_loss)
        history['t_f1'].append(t_f1); history['v_f1'].append(v_f1)
        
        print(f"Ep {epoch+1}: T_Loss {t_loss:.3f} | T_F1 {t_f1:.3f} | V_Loss {v_loss:.3f} | V_F1 {v_f1:.3f}")
        
        if v_f1 > best_f1:
            best_f1 = v_f1
            torch.save(model.state_dict(), f"fold_{fold}_best.pth")
            
    print(f"Best Val F1: {best_f1:.4f}")
    return history, v_preds, v_targets

# --- 6. EXECUTION ---
def run(df_final):
    class DummyLabelEncoder:
        def __init__(self):
            self.classes_ = ['Group 0 (Luminal A/B)', 'Group 1 (HER2+/TN)']
            
        def inverse_transform(self, predictions):
            return [self.classes_[p] for p in predictions]
    
    le = DummyLabelEncoder()

    df = df_final
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    histories, all_targets, all_preds = [], [], []
    
    for fold, (t_idx, v_idx) in enumerate(skf.split(df, df['label_binary'])):
        h, p, t = train_fold(fold, t_idx, v_idx, df, le, PRECOMPUTED_DIR)
        histories.append(h)
        all_preds.extend(p)
        all_targets.extend(t)
        
    # Plots
    fig, axes = plt.subplots(N_FOLDS, 2, figsize=(12, 4*N_FOLDS))
    for i, h in enumerate(histories):
        axes[i][0].plot(h['t_loss'], label='Train')
        axes[i][0].plot(h['v_loss'], label='Val')
        axes[i][0].legend(); axes[i][0].set_title(f'Fold {i+1} Loss')
        axes[i][1].plot(h['t_f1'], label='Train F1')
        axes[i][1].plot(h['v_f1'], label='Val F1')
        axes[i][1].legend(); axes[i][1].set_title(f'Fold {i+1} F1')
    plt.tight_layout()
    plt.show()
    
    cm = confusion_matrix(all_targets, all_preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.show()
    return le




In [2]:
# 1. Initial Setup (Call once)
DF_FINAL = setup_data_and_preprocess()

--- 🧠 Data Setup & Preprocessing Started ---
Starting precomputation and saving of processed ROIs...


Preprocessing Images: 100%|██████████| 581/581 [01:15<00:00,  7.71it/s]


--- ✅ Precomputation Complete. Final images available: 581 ---


In [8]:
le = run(DF_FINAL)



=== FOLD 1/5 ===


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 138MB/s] 


Ep 1: T_Loss 0.699 | T_F1 0.508 | V_Loss 0.697 | V_F1 0.455


Ep 2: T_Loss 0.681 | T_F1 0.548 | V_Loss 0.699 | V_F1 0.442


Ep 3: T_Loss 0.685 | T_F1 0.488 | V_Loss 0.710 | V_F1 0.504


Ep 4: T_Loss 0.680 | T_F1 0.552 | V_Loss 0.706 | V_F1 0.488


Ep 5: T_Loss 0.667 | T_F1 0.590 | V_Loss 0.702 | V_F1 0.552


Ep 6: T_Loss 0.671 | T_F1 0.588 | V_Loss 0.712 | V_F1 0.522


KeyboardInterrupt: 

# Tuna salad

In [ ]:
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold
import torch.optim as optim
import numpy as np

# --- 8. OPTUNA OBJECTIVE FUNCTION ---
# This function wraps the training logic for Optuna to optimize.
def objective(trial):
    # Hyperparameter search space
    
    # 1. Learning Rate (log-uniform is standard for LR)
    LR = trial.suggest_loguniform('lr', 1e-5, 5e-4, log=True)
    
    # 2. Weight Decay (L2 Regularization)
    WEIGHT_DECAY = trial.suggest_loguniform('weight_decay', 1e-4, 1e-2, log=True)
    
    # 3. MixUp Alpha (Intensity of generalization)
    MIXUP_ALPHA = trial.suggest_uniform('mixup_alpha', 0.2, 0.8, log=True)
    
    # Define optimization settings
    N_OPTI_FOLDS = 2  # Set this to 1 or 2 as desired for speed

    N_OPTI_EPOCHS = 8  # Use 8 epochs instead of the final 25
    
    # --- Data and Setup (Reusing code from run()) ---
    df = pd.read_csv(PATHS['labels'])
    valid_mask = df['sample_index'].apply(lambda x: os.path.exists(os.path.join(PATHS['train_img'], x)))
    df = df[valid_mask].reset_index(drop=True)

    # Recreate binary labels (assuming the fix for GROUP_0/GROUP_1 is applied)
    GROUP_0 = ['Luminal A', 'Luminal B']
    GROUP_1 = ['HER2(+)', 'Triple Negative'] 
    
    def get_binary_label(original_label):
        if original_label in GROUP_0:
            return 0
        elif original_label in GROUP_1:
            return 1
        else:
            return -1 # Should not happen with corrected labels
            
    df['label_binary'] = df['label'].apply(get_binary_label)
    df['label_encoded'] = df['label_binary']
    
    # Dummy LabelEncoder is needed for F1 score plotting/context
    class DummyLabelEncoder:
        def __init__(self):
            self.classes_ = ['Group 0 (Luminal A/B)', 'Group 1 (HER2+/TN)']
        def inverse_transform(self, predictions):
            return [self.classes_[p] for p in predictions]
            
    le = DummyLabelEncoder()
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    
    # Use a fraction of the folds for faster optimization
    folds_to_use = list(skf.split(df, df['label_binary']))[:N_OPTI_FOLDS]
    
    fold_f1s = []
    
    for fold, (t_idx, v_idx) in enumerate(folds_to_use):
        
        # --- TRAINING LOGIC (Modified train_fold content) ---
        train_df = df.iloc[t_idx].reset_index(drop=True)
        val_df = df.iloc[v_idx].reset_index(drop=True)

        # Class Balancing (using fixed MIXUP_ALPHA)
        class_counts = train_df['label_encoded'].value_counts().sort_index().values
        class_weights = 1. / class_counts
        sample_weights = [class_weights[l] for l in train_df['label_encoded']]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

        # IMPORTANT: We need a new get_transforms() call that uses the updated IMG_SIZE
        # Ensure your external get_transforms() function is correctly using the global IMG_SIZE
        train_loader = DataLoader(TissueDataset(train_df, PATHS['train_img'], PATHS['train_mask'], get_transforms()[0]), 
                                  batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
        val_loader = DataLoader(TissueDataset(val_df, PATHS['train_img'], PATHS['train_mask'], get_transforms()[1]), 
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        model = get_model(num_classes = 1)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.BCEWithLogitsLoss()
        #scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        
        # Training loop is exactly the same as in your train_fold, but with 
        # the optimized LR, WD, and the new MIXUP_ALPHA (passed via the global scope)
        best_f1 = 0.0
        
        for epoch in range(N_OPTI_EPOCHS):
            model.train()
            # Training phase
            t_preds, t_targets = [], []
            for imgs, lbls in train_loader:
                # ... (standard training loop content)
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                optimizer.zero_grad()
                
                # --- MIXUP BLOCK ---
                lbls_float = lbls.float().unsqueeze(1)
                # Pass the trial's MIXUP_ALPHA
                imgs, targets_a, targets_b, lam = mixup_data(imgs, lbls_float, MIXUP_ALPHA) 
                
                out = model(imgs)
                loss = mixup_criterion(criterion, out, targets_a, targets_b, lam)
                loss.backward()
                optimizer.step()
                
                # Metrics accumulation (essential for the F1 score for the epoch)
                t_preds.extend((torch.sigmoid(out) >= 0.5).long().squeeze().cpu().numpy())
                t_targets.extend(targets_a.round().long().squeeze().cpu().numpy())
                
            #scheduler.step()
            
            # Validation phase
            model.eval()
            v_preds, v_targets = [], []
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                    lbls_float = lbls.float().unsqueeze(1)
                    out = model(imgs)
                    
                    v_preds.extend((torch.sigmoid(out) >= 0.5).long().squeeze().cpu().numpy())
                    v_targets.extend(lbls.cpu().numpy())
            
            # Calculate F1 for this epoch
            v_f1 = f1_score(v_targets, v_preds, average='macro')
            
            if v_f1 > best_f1:
                best_f1 = v_f1
                # OPTUNA PRUNING: Report intermediate result
                # trial.report(best_f1, epoch) # Optionally add pruning later
                
        # Store the best F1 from this fold
        fold_f1s.append(best_f1)
        
    # Optuna aims to maximize the mean F1 score across the N_OPTI_FOLDS
    return np.mean(fold_f1s)

# --- 9. OPTUNA EXECUTION ---
def run_optuna(n_trials=50):
    # sampler=TPESampler(seed=42) uses Tree-structured Parzen Estimator, good for efficiency
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
    print(f"Starting Optuna search for {n_trials} trials...")
    study.optimize(objective, n_trials=n_trials)

    print("\n--- OPTUNA RESULTS ---")
    print(f"Best trial value (Mean Macro F1): {study.best_value:.4f}")
    print("Best hyperparameters found:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    return study.best_params



In [ ]:
# Example of how to run the optimization:
best_params = run_optuna(n_trials=50) # Run for 50 combinations
print(best_params)

In [ ]:
# --- 7. SUBMISSION (TTA 4x) ---
def submit(le):
    print("Generating Submission with TTA...")
    test_files = [f for f in os.listdir(PATHS['test_img']) if f.endswith('.png')]
    test_df = pd.DataFrame({'filename': test_files})
    
    # Dataset di test base
    test_ds = TissueDataset(test_df, PATHS['test_img'], PATHS['test_mask'], get_transforms()[1], is_test=True)
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    models_list = []
    for i in range(N_FOLDS):
        m = get_model(len(le.classes_))
        m.load_state_dict(torch.load(f"fold_{i}_best.pth"))
        m.eval()
        models_list.append(m)
        
    preds, fnames = [], []
    
    with torch.no_grad():
        for imgs, names in tqdm(loader):
            imgs = imgs.to(DEVICE)
            
            # --- TTA: 4 Viste (Orig, FlipH, FlipV, Rot90) ---
            imgs_lr = torch.flip(imgs, [3])
            imgs_ud = torch.flip(imgs, [2])
            imgs_rot = torch.rot90(imgs, 1, [2, 3])
            
            inputs = [imgs, imgs_lr, imgs_ud, imgs_rot]
            
            batch_probs = torch.zeros((imgs.size(0), len(le.classes_))).to(DEVICE)
            
            # Per ogni modello (Ensemble)
            for m in models_list:
                # Per ogni vista (TTA)
                for inp in inputs:
                    batch_probs += torch.softmax(m(inp), dim=1)
            
            # Media totale: 5 Fold * 4 TTA = 20 predizioni per immagine
            batch_probs /= (len(models_list) * len(inputs))
            
            preds.extend(le.inverse_transform(batch_probs.argmax(1).cpu().numpy()))
            fnames.extend(names)
            
    pd.DataFrame({'sample_index': fnames, 'label': preds}).to_csv('submission.csv', index=False)
    print("Done.")


In [ ]:
#submit(le)